# Machine Learning — Lab 3
## Preprocessing Pipelines and Feature Engineering

**Main Course Learning Outcome — CLO2**  
**CLO2:** Analyze datasets and apply appropriate preprocessing, transformation, and feature engineering techniques.

**Environment:** Python 3 / Jupyter Notebook  
**Libraries:** `numpy`, `pandas`, `matplotlib`, `scikit-learn`

> **Assessment principle:** Full credit requires more than producing transformed arrays. You must explain **why each transformation is needed, where it should be fitted, what information it uses, and how it changes the representation available to a model**.

## Lab at a Glance

| Stage | Suggested time | What you will do |
|---|---:|---|
| 1. Inspect feature roles | 10 min | Separate numerical, categorical, identifier, and target columns |
| 2. Split before preprocessing | 15 min | Create reproducible train/validation/test sets |
| 3. Numerical preprocessing | 25 min | Impute, standardize, normalize, and transform skewed features |
| 4. Categorical preprocessing | 20 min | Clean and one-hot encode categories |
| 5. Feature engineering | 25 min | Create domain-informed derived and interaction features |
| 6. Build a pipeline | 15 min | Use `ColumnTransformer` and `Pipeline` correctly |
| 7. Leakage/debugging challenge | 10 min | Detect an incorrect preprocessing workflow |
| **Total** | **120 min** | |

### Main idea

A model learns from the **representation** it receives.

$$
\text{Raw Data}
\rightarrow
\text{Clean}
\rightarrow
\text{Transform}
\rightarrow
\text{Engineer Features}
\rightarrow
\text{Model-Ready Data}
$$

## Learning Objectives

By the end of this lab, you should be able to:

1. distinguish numerical, categorical, identifier, and target columns;
2. explain why preprocessing must be fitted on training data only;
3. apply median imputation to numerical features;
4. standardize numerical features using training-set statistics;
5. apply one-hot encoding to nominal categorical variables;
6. handle unseen categories safely;
7. use log transformation for positively skewed features;
8. construct simple domain-informed features;
9. build a reusable `ColumnTransformer` preprocessing pipeline;
10. diagnose preprocessing leakage and inconsistent transformations.

# Part I — Dataset and Problem Context

We will use a synthetic **student engagement dataset** designed for this lab.

The task is not to build a sophisticated classifier yet. Instead, the goal is to prepare a valid feature matrix for a later supervised-learning model.

The target is:

```text
high_performance
```

where:

- `1` = high performance;
- `0` = lower performance.

The dataset contains:

- numerical features;
- categorical features;
- missing values;
- a skewed activity feature;
- an identifier;
- one deliberately dangerous leakage feature.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, FunctionTransformer

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

print("Machine Learning Lab 3 environment ready.")

In [ ]:
rng = np.random.default_rng(3452)
n = 320

student_id = np.arange(20001, 20001 + n)

study_hours = np.clip(rng.normal(8.0, 3.2, size=n), 0.5, 20.0)
attendance_pct = np.clip(rng.normal(81.0, 12.0, size=n), 35.0, 100.0)
previous_gpa = np.clip(rng.normal(2.9, 0.55, size=n), 1.2, 4.0)
commute_minutes = np.clip(rng.gamma(shape=2.1, scale=13.0, size=n), 2.0, 120.0)
weekly_lms_minutes = np.clip(rng.lognormal(mean=5.0, sigma=0.75, size=n), 10.0, 1500.0)
missed_labs = np.clip(rng.poisson(2.0, size=n), 0, 8)

learning_mode = rng.choice(
    ["Individual", "Group", "Online"],
    size=n,
    p=[0.38, 0.34, 0.28]
)

faculty = rng.choice(
    ["Computing", "Engineering", "Science"],
    size=n,
    p=[0.45, 0.32, 0.23]
)

scholarship = rng.choice(["Yes", "No"], size=n, p=[0.42, 0.58])

latent = (
    1.9 * study_hours
    + 0.28 * attendance_pct
    + 8.5 * previous_gpa
    + 0.010 * weekly_lms_minutes
    - 2.5 * missed_labs
    - 0.05 * commute_minutes
    + np.where(learning_mode == "Group", 3.0, 0.0)
    + rng.normal(0, 8, size=n)
)

high_performance = (latent >= np.median(latent)).astype(int)

df_master = pd.DataFrame({
    "student_id": student_id,
    "study_hours": np.round(study_hours, 1),
    "attendance_pct": np.round(attendance_pct, 1),
    "previous_gpa": np.round(previous_gpa, 2),
    "commute_minutes": np.round(commute_minutes, 1),
    "weekly_lms_minutes": np.round(weekly_lms_minutes, 1),
    "missed_labs": missed_labs,
    "learning_mode": learning_mode,
    "faculty": faculty,
    "scholarship": scholarship,
    "high_performance": high_performance,
})

# Missing values.
for col, count in {
    "attendance_pct": 16,
    "previous_gpa": 12,
    "commute_minutes": 10,
    "faculty": 8,
}.items():
    rows = rng.choice(df_master.index, size=count, replace=False)
    df_master.loc[rows, col] = np.nan

# Slightly inconsistent categories.
rows = rng.choice(df_master.index, size=12, replace=False)
variants = [" group ", "ONLINE", "individual", "Group ", " online", "INDIVIDUAL"] * 2
for idx, val in zip(rows, variants):
    df_master.loc[idx, "learning_mode"] = val

# A leakage feature that would only be known after the course ends.
df_master["final_result_recorded"] = np.where(
    df_master["high_performance"] == 1,
    "High",
    "Low"
)

print("Master dataset shape:", df_master.shape)
display(df_master.head())

## Task 1.1 — Personalized Working Dataset

Enter the **last four digits** of your student ID.

Your student ID determines a reproducible sample of 240 observations.

In [ ]:
# TODO: Replace None with the last four digits of your own student ID.
STUDENT_ID_LAST4 = None

if STUDENT_ID_LAST4 is None:
    raise ValueError("Enter the last four digits of your student ID.")

if not isinstance(STUDENT_ID_LAST4, int):
    raise TypeError("STUDENT_ID_LAST4 must be an integer.")

SEED = 3000 + (STUDENT_ID_LAST4 % 7000)

df = df_master.sample(
    n=240,
    random_state=SEED,
    replace=False
).reset_index(drop=True)

print("Your preprocessing seed:", SEED)
print("Working dataset shape:", df.shape)

## Task 1.2 — Identify Feature Roles

Before running any preprocessing, classify the columns.

| Column | Numerical / Categorical / Identifier / Target | Keep as feature? | Why? |
|---|---|---|---|
| `student_id` |  |  |  |
| `study_hours` |  |  |  |
| `attendance_pct` |  |  |  |
| `previous_gpa` |  |  |  |
| `commute_minutes` |  |  |  |
| `weekly_lms_minutes` |  |  |  |
| `missed_labs` |  |  |  |
| `learning_mode` |  |  |  |
| `faculty` |  |  |  |
| `scholarship` |  |  |  |
| `final_result_recorded` |  |  |  |
| `high_performance` |  |  |  |

### Required reasoning

Explain why both `student_id` and `final_result_recorded` should normally be excluded, but for different reasons.

# Part II — Split Before Preprocessing

The correct workflow is:

$$
\text{Raw Data}
\rightarrow
\text{Split}
\rightarrow
\text{Fit Preprocessing on Training Only}
\rightarrow
\text{Transform Validation/Test}
$$

The incorrect workflow is:

$$
\text{Raw Data}
\rightarrow
\text{Fit Preprocessing on All Data}
\rightarrow
\text{Split}.
$$

The second workflow allows validation/test information to influence the preprocessing parameters.

In [ ]:
target_col = "high_performance"

excluded_cols = [
    "student_id",
    "final_result_recorded",
    target_col,
]

X = df.drop(columns=excluded_cols)
y = df[target_col].copy()

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.40,
    random_state=SEED,
    stratify=y,
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp,
)

print("Training:", X_train.shape)
print("Validation:", X_valid.shape)
print("Test:", X_test.shape)

## Task 2.1 — Predict Before Checking

Before the next cell:

1. What proportion of observations should be in training?
2. Which preprocessing statistics must come only from training?
3. Give three examples of such learned statistics.
4. Why should category information from the test set not change the fitted encoder?

**Your answer:**

In [ ]:
print("Training target distribution:")
display(y_train.value_counts(normalize=True).sort_index().rename("proportion"))

print("\nValidation target distribution:")
display(y_valid.value_counts(normalize=True).sort_index().rename("proportion"))

print("\nTest target distribution:")
display(y_test.value_counts(normalize=True).sort_index().rename("proportion"))

# Part III — Numerical Preprocessing

We will focus on:

- missing-value imputation;
- standardization;
- min-max normalization;
- log transformation.

Different transformations serve different purposes.

## Task 3.1 — Inspect Missing Numerical Values

Predict which numerical columns contain missing values, then run:

In [ ]:
numeric_cols = [
    "study_hours",
    "attendance_pct",
    "previous_gpa",
    "commute_minutes",
    "weekly_lms_minutes",
    "missed_labs",
]

display(
    X_train[numeric_cols]
    .isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)

## Task 3.2 — Median Imputation

Why might median imputation be preferable to mean imputation for skewed variables such as `commute_minutes`?

Answer before running the next cell.

In [ ]:
median_imputer = SimpleImputer(strategy="median")

median_imputer.fit(X_train[numeric_cols])

train_num_imputed = pd.DataFrame(
    median_imputer.transform(X_train[numeric_cols]),
    columns=numeric_cols,
    index=X_train.index
)

valid_num_imputed = pd.DataFrame(
    median_imputer.transform(X_valid[numeric_cols]),
    columns=numeric_cols,
    index=X_valid.index
)

print("Learned training-set medians:")
display(pd.Series(
    median_imputer.statistics_,
    index=numeric_cols,
    name="training_median"
).round(3))

print("\nRemaining missing values in transformed training data:",
      train_num_imputed.isna().sum().sum())
print("Remaining missing values in transformed validation data:",
      valid_num_imputed.isna().sum().sum())

## Task 3.3 — Why Training Medians?

Choose one imputed feature and compare:

- training median;
- validation median;
- full-data median.

Then answer:

1. Why is the training median the only one that should be used by the fitted imputer?
2. What information would leak if the full-data median were used?
3. Would the numerical difference always be large?
4. Why is leakage still conceptually wrong even when the difference is small?

In [ ]:
comparison_feature = "attendance_pct"

print("Training median:",
      X_train[comparison_feature].median())
print("Validation median:",
      X_valid[comparison_feature].median())
print("Full working-data median:",
      X[comparison_feature].median())

## Task 3.4 — Standardization

Standardization uses:

$$
z=\frac{x-\mu}{\sigma}.
$$

Complete the code below so that the scaler is fitted **only on the imputed training data**.

In [ ]:
scaler = StandardScaler()

# TODO: fit the scaler on train_num_imputed.
# scaler.fit(...)

# TODO: transform training and validation data.
train_num_scaled = None
valid_num_scaled = None

### Self-check

After completing Task 3.4, run:

In [ ]:
if train_num_scaled is None or valid_num_scaled is None:
    raise ValueError("Complete the standardization step first.")

train_num_scaled_df = pd.DataFrame(
    train_num_scaled,
    columns=numeric_cols,
    index=X_train.index
)

valid_num_scaled_df = pd.DataFrame(
    valid_num_scaled,
    columns=numeric_cols,
    index=X_valid.index
)

train_means = train_num_scaled_df.mean()
train_stds = train_num_scaled_df.std(ddof=0)

print("Training means after standardization:")
display(train_means.round(6).to_frame("mean"))

print("Training std after standardization:")
display(train_stds.round(6).to_frame("std"))

assert np.allclose(train_means.values, 0, atol=1e-8)
assert np.allclose(train_stds.values, 1, atol=1e-8)

print("Standardization checks passed.")

## Task 3.5 — Validation Data Is Not Forced to Mean 0

Inspect the validation means after transformation.

Predict first: should every validation feature have mean exactly 0?

Explain why or why not.

In [ ]:
print("Validation means after applying TRAINING scaler:")
display(valid_num_scaled_df.mean().round(3).to_frame("validation_mean"))

## Task 3.6 — Min-Max Normalization

Min-max scaling uses:

$$
x'=\frac{x-x_{\min}}{x_{\max}-x_{\min}}.
$$

Fit a `MinMaxScaler` on the **training data only** and compare with standardization.

Answer:

1. Which transformation gives approximately mean 0 and standard deviation 1?
2. Which transformation maps the training range approximately to $[0,1]$?
3. Which transformation is more sensitive to extreme values?
4. Why can validation/test values sometimes fall outside $[0,1]$?

In [ ]:
minmax = MinMaxScaler()
minmax.fit(train_num_imputed)

train_num_minmax = pd.DataFrame(
    minmax.transform(train_num_imputed),
    columns=numeric_cols,
    index=X_train.index
)

valid_num_minmax = pd.DataFrame(
    minmax.transform(valid_num_imputed),
    columns=numeric_cols,
    index=X_valid.index
)

display(train_num_minmax.describe().loc[["min", "max"]].round(3))

## Task 3.7 — Log Transformation for Skewed Features

The feature `weekly_lms_minutes` is positively skewed.

We can use:

$$
x'=\log(1+x).
$$

Before running the next cell:

1. Predict what will happen to very large values.
2. Predict whether the range will become more compressed.
3. Explain why $\log(1+x)$ is convenient when $x=0$.

In [ ]:
feature = "weekly_lms_minutes"

fig = plt.figure(figsize=(7, 4))
plt.hist(X_train[feature].dropna(), bins=18, edgecolor="black")
plt.xlabel(feature)
plt.ylabel("Frequency")
plt.title("Before Log Transformation")
plt.show()

logged = np.log1p(X_train[feature].dropna())

fig = plt.figure(figsize=(7, 4))
plt.hist(logged, bins=18, edgecolor="black")
plt.xlabel(f"log1p({feature})")
plt.ylabel("Frequency")
plt.title("After Log Transformation")
plt.show()

## Task 3.8 — Interpret the Log Transformation

Answer:

1. Did the large values become less dominant?
2. Did the transformation preserve the ordering of observations?
3. Would it be appropriate for a feature containing negative values?
4. Why should a transformation be justified by data characteristics rather than applied automatically?

# Part IV — Categorical Preprocessing

Nominal categories should not usually be replaced by arbitrary integers such as:

```text
Individual -> 0
Group      -> 1
Online     -> 2
```

because that creates a false ordering.

We will:

1. standardize category text;
2. impute missing categories;
3. one-hot encode.

## Task 4.1 — Clean Category Text

Complete the function so that:

```text
" group "      -> "Group"
"ONLINE"       -> "Online"
"individual"   -> "Individual"
```

In [ ]:
def clean_learning_mode(series):
    # TODO: strip whitespace and normalize capitalization.
    cleaned = None
    return cleaned

In [ ]:
toy = pd.Series([" group ", "ONLINE", "individual", "Group"])
toy_clean = clean_learning_mode(toy)

assert toy_clean.tolist() == ["Group", "Online", "Individual", "Group"]

print("Category cleaning test passed.")

In [ ]:
X_train_cat = X_train.copy()
X_valid_cat = X_valid.copy()
X_test_cat = X_test.copy()

for frame in [X_train_cat, X_valid_cat, X_test_cat]:
    frame["learning_mode"] = clean_learning_mode(frame["learning_mode"])

print("Training learning_mode values:")
print(sorted(X_train_cat["learning_mode"].dropna().unique()))

## Task 4.2 — One-Hot Encoding

Categorical columns:

```text
learning_mode
faculty
scholarship
```

Predict:

1. Why does one-hot encoding avoid false numerical order?
2. What happens to dimensionality?
3. Why is `handle_unknown="ignore"` useful for validation/test data?

In [ ]:
categorical_cols = [
    "learning_mode",
    "faculty",
    "scholarship",
]

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

categorical_pipeline.fit(X_train_cat[categorical_cols])

train_cat_encoded = categorical_pipeline.transform(
    X_train_cat[categorical_cols]
)

valid_cat_encoded = categorical_pipeline.transform(
    X_valid_cat[categorical_cols]
)

feature_names_cat = categorical_pipeline.named_steps["onehot"].get_feature_names_out(categorical_cols)

print("Encoded categorical feature names:")
print(feature_names_cat)

print("\nTraining encoded shape:", train_cat_encoded.shape)
print("Validation encoded shape:", valid_cat_encoded.shape)

## Task 4.3 — Unseen Category Challenge

Suppose a future validation example contains:

```text
faculty = "Business"
```

but `"Business"` never appeared in the training data.

Predict what `handle_unknown="ignore"` should do before running the next cell.

In [ ]:
future_example = X_valid_cat[categorical_cols].iloc[[0]].copy()
future_example["faculty"] = "Business"

future_encoded = categorical_pipeline.transform(future_example)

print("Future encoded shape:", future_encoded.shape)
print("Number of nonzero encoded values:", int((future_encoded != 0).sum()))

## Task 4.4 — Interpret Unknown Categories

Answer:

1. Did the transform crash?
2. Was a new `"faculty_Business"` column created?
3. Why would creating a new column at prediction time be dangerous?
4. What does the all-zero pattern for unseen faculty categories imply?
5. What other handling strategies could be used in a production system?

# Part V — Feature Engineering

Feature engineering creates new variables from existing data.

A useful engineered feature should have:

- a clear interpretation;
- information available at prediction time;
- no target leakage;
- a defensible relationship to the problem.

## Task 5.1 — Construct an Engagement Index

Create:

$$
\text{engagement\_index}
=
0.4\times\frac{\text{attendance}}{100}
+
0.4\times\frac{\text{study hours}}{20}
+
0.2\times\left(1-\frac{\text{missed labs}}{8}\right).
$$

This is not a learned model. It is a manually designed feature.

Complete the function.

In [ ]:
def add_engagement_index(frame):
    result = frame.copy()

    # TODO: implement the formula above.
    result["engagement_index"] = None

    return result

In [ ]:
toy_frame = pd.DataFrame({
    "attendance_pct": [100.0],
    "study_hours": [20.0],
    "missed_labs": [0],
})

toy_engineered = add_engagement_index(toy_frame)

assert abs(toy_engineered["engagement_index"].iloc[0] - 1.0) < 1e-9

print("Feature-engineering test passed.")

## Task 5.2 — Interpret the Engineered Feature

Answer:

1. What does a larger `engagement_index` represent?
2. Why is this feature interpretable?
3. Which assumptions are embedded in the weights 0.4, 0.4, and 0.2?
4. Is this feature guaranteed to improve a future model?
5. How should that question eventually be answered?

**Your answers:**

## Task 5.3 — Interaction Feature

Create an interaction:

$$
\text{study\_attendance}
=
\text{study\_hours}
\times
\frac{\text{attendance\_pct}}{100}.
$$

Interpret this as a rough measure of study effort adjusted by attendance.

Complete:

In [ ]:
def add_interaction(frame):
    result = frame.copy()

    # TODO: create study_attendance.
    result["study_attendance"] = None

    return result

In [ ]:
toy_interaction = pd.DataFrame({
    "study_hours": [10.0],
    "attendance_pct": [80.0],
})

toy_interaction = add_interaction(toy_interaction)
assert abs(toy_interaction["study_attendance"].iloc[0] - 8.0) < 1e-9

print("Interaction feature test passed.")

## Task 5.4 — Leakage Check for Feature Engineering

Which of the following would be valid **before the final result is known**?

| Candidate feature | Valid? | Why? |
|---|---|---|
| `study_hours * attendance_pct` |  |  |
| `quiz_average_to_date` |  |  |
| `final_result_recorded == "High"` |  |  |
| `number_of_missed_labs_so_far` |  |  |
| `final_exam_score / 100` |  |  |

### Rule

A feature is not valid merely because it is mathematically easy to compute.

It must also be available at the intended prediction time.

# Part VI — Build a Complete Preprocessing Pipeline

We now combine numerical and categorical transformations into one reusable object.

The pipeline will:

- impute numerical values using medians;
- standardize numerical values;
- impute categorical values using the most frequent category;
- one-hot encode categorical values;
- ignore unseen categories safely.

The pipeline must be fitted on **training data only**.

In [ ]:
# First, clean category text in all splits using a deterministic text-cleaning rule.
X_train_p = X_train.copy()
X_valid_p = X_valid.copy()
X_test_p = X_test.copy()

for frame in [X_train_p, X_valid_p, X_test_p]:
    frame["learning_mode"] = clean_learning_mode(frame["learning_mode"])

numerical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline_full = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_pipeline, numeric_cols),
        ("cat", categorical_pipeline_full, categorical_cols),
    ],
    remainder="drop",
)

# TODO: Fit the preprocessor on X_train_p only.
# preprocessor.fit(...)

# TODO: transform train, validation, and test.
X_train_ready = None
X_valid_ready = None
X_test_ready = None

### Self-check

Run after completing the pipeline.

In [ ]:
if X_train_ready is None or X_valid_ready is None or X_test_ready is None:
    raise ValueError("Complete the preprocessing pipeline first.")

assert X_train_ready.shape[0] == len(X_train_p)
assert X_valid_ready.shape[0] == len(X_valid_p)
assert X_test_ready.shape[0] == len(X_test_p)

assert X_train_ready.shape[1] == X_valid_ready.shape[1] == X_test_ready.shape[1]

print("Pipeline output shapes:")
print("Train:", X_train_ready.shape)
print("Validation:", X_valid_ready.shape)
print("Test:", X_test_ready.shape)

print("Pipeline shape checks passed.")

In [ ]:
feature_names_all = preprocessor.get_feature_names_out()

print("First 20 transformed feature names:")
for name in feature_names_all[:20]:
    print(name)

print("\nTotal transformed features:", len(feature_names_all))

## Task 6.1 — Explain the Final Representation

Answer:

1. Why does the transformed dataset contain more columns than the original feature table?
2. Which transformations learned numerical statistics from training data?
3. Which transformation learned category levels from training data?
4. Why do validation and test have exactly the same transformed columns as training?
5. Why is the preprocessing object itself part of the trained ML system?

# Part VII — Deliberate Leakage and Debugging

Consider this incorrect code:

```python
bad_scaler = StandardScaler()
bad_scaler.fit(X[numeric_cols])   # all data
X_all_scaled = bad_scaler.transform(X[numeric_cols])

# split after scaling
```

This code may run perfectly.

But it is still wrong.

## Task 7.1 — Explain the Leakage

Answer:

1. What information from validation/test data enters the fitted scaler?
2. Which quantities are leaked?
3. Why might the numerical effect be small but the workflow still invalid?
4. What is the correct sequence?

## Task 7.2 — Compare Good and Bad Scaling

Run the demonstration below and compare the learned mean for one feature.

In [ ]:
bad_scaler = StandardScaler()
bad_scaler.fit(X[numeric_cols].fillna(X[numeric_cols].median()))

good_feature = "attendance_pct"

good_mean = scaler.mean_[numeric_cols.index(good_feature)]
bad_mean = bad_scaler.mean_[numeric_cols.index(good_feature)]

print(f"Training-only mean for {good_feature}: {good_mean:.4f}")
print(f"Full-data mean for {good_feature}:     {bad_mean:.4f}")
print(f"Absolute difference:                  {abs(good_mean - bad_mean):.4f}")

## Task 7.3 — Deliberate Debugging

The code below transforms validation data by fitting a **new scaler**:

```python
valid_scaler = StandardScaler()
X_valid_wrong = valid_scaler.fit_transform(valid_num_imputed)
```

Explain why this is wrong.

Then write the correct one-line transformation below.

In [ ]:
# TODO: transform validation data using the already fitted training scaler.
X_valid_correct = None

In [ ]:
if X_valid_correct is None:
    raise ValueError("Complete X_valid_correct.")

assert np.allclose(X_valid_correct, scaler.transform(valid_num_imputed))

print("Validation transformation is correct.")

# Part VIII — Personalized Transformation Challenge

Your student-ID seed assigns one preprocessing question.

Run:

In [ ]:
challenges = [
    "A numerical feature has 8% missing values and strong right skew.",
    "A nominal feature has 5 categories and 3% missing values.",
    "A numerical feature ranges from 0 to 1 while another ranges from 0 to 200000.",
    "A categorical feature contains 200 nearly unique IDs.",
]

assigned_challenge = challenges[SEED % len(challenges)]

print("Your assigned challenge:")
print(assigned_challenge)

## Task 8.1 — Design the Transformation

For your assigned challenge, write:

- the data problem;
- the preprocessing or feature-engineering step you recommend;
- what should be **fitted** on training data;
- what can be applied deterministically without fitting;
- one risk or limitation of your choice;
- how you would check whether the transformation is useful.

**Your answer:**

## Task 8.2 — Prediction Before Modification

Choose one transformation used in this lab and remove it mentally.

Predict what would happen if we removed:

- imputation;
- scaling;
- category cleaning;
- one-hot encoding;
- `handle_unknown="ignore"`.

Then explain which later model families would be most affected and why.

**Your prediction:**

# Individual Understanding Check

Your instructor may select one question for a 60–90 second explanation.

1. Why must preprocessing be fitted on training data only?
2. What is the difference between standardization and min-max normalization?
3. Why is one-hot encoding appropriate for nominal categories?
4. Why can unseen categories appear in validation/test data?
5. Give one example of a valid engineered feature and one example of leakage.
6. Why is a preprocessing pipeline part of the final ML system?
7. In your personalized challenge, defend your chosen transformation.

You should be able to answer without reading a prepared paragraph.

# Reflection

Answer concisely in your own words.

1. Which preprocessing step in this lab actually **learned** information from the training data?
2. Which preprocessing step was purely deterministic?
3. Why can the same raw dataset produce a very different feature matrix after one-hot encoding?
4. Why is feature engineering not guaranteed to improve performance?
5. What is the single most important leakage rule from this lab?

**Your reflection:**

# Submission Checklist

Before submitting, confirm that your notebook contains:

- [ ] your own student-ID-derived sample;
- [ ] feature-role classification;
- [ ] correct train/validation/test split;
- [ ] median-imputation analysis;
- [ ] completed standardization task;
- [ ] min-max comparison;
- [ ] log-transformation interpretation;
- [ ] completed category-cleaning function;
- [ ] one-hot encoding and unseen-category analysis;
- [ ] completed engagement-index feature;
- [ ] completed interaction feature;
- [ ] completed full `ColumnTransformer` pipeline;
- [ ] leakage comparison;
- [ ] corrected validation-scaling debugging task;
- [ ] personalized transformation challenge;
- [ ] reflection answers;
- [ ] all required code cells executed successfully.

# Assessment — 10 Marks

| Component | Marks |
|---|---:|
| Correct implementation | **2** |
| Data-analysis / preprocessing justification | **3** |
| Experimental analysis | **2** |
| Prediction / debugging evidence | **1** |
| Individual understanding check | **1** |
| Code quality and submission completeness | **1** |
| **Total** | **10** |

### Marking emphasis

Full marks require showing that you understand:

$$
\text{what is transformed}
\rightarrow
\text{why it is transformed}
\rightarrow
\text{what is fitted}
\rightarrow
\text{where it is fitted}
\rightarrow
\text{how the representation changes}.
$$

# Lab 3 Summary

You should now be able to build a model-ready representation without leaking information:

$$
\boxed{
\text{Split}
\rightarrow
\text{Fit preprocessing on training}
\rightarrow
\text{Transform validation/test}
\rightarrow
\text{Engineer valid features}
}
$$

### Key lessons

- Imputation statistics must come from training data.
- Scaling parameters must come from training data.
- One-hot encoders must be fitted on training categories.
- Validation/test data may contain unseen categories.
- Feature engineering should be interpretable and available at prediction time.
- Pipelines make preprocessing reproducible and consistent.
- Correct preprocessing is part of machine learning, not a preliminary clerical step.

**Next lab:** Linear Regression and Residual Analysis.